In [11]:
import pandas as pd
import numpy as np

In [12]:
telemetry = pd.read_csv(
    "../data_pack/raw/turbine_telemetry_14day_sample.csv"
)

dam = pd.read_csv(
    "../data_pack/raw/dam_price_14day_sample.csv"
)

In [13]:
print(telemetry.info())

print(dam.info())

<class 'pandas.DataFrame'>
RangeIndex: 1680 entries, 0 to 1679
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   timestamp     1680 non-null   str    
 1   turbine_id    1680 non-null   str    
 2   wind_speed    1680 non-null   float64
 3   power_kw      1680 non-null   float64
 4   availability  1680 non-null   float64
dtypes: float64(3), str(2)
memory usage: 101.8 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 336 entries, 0 to 335
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   timestamp              336 non-null    str    
 1   dam_price_inr_per_kwh  336 non-null    float64
dtypes: float64(1), str(1)
memory usage: 11.6 KB
None


In [14]:
telemetry["timestamp"] = pd.to_datetime(
    telemetry["timestamp"]
)

dam["timestamp"] = pd.to_datetime(
    dam["timestamp"]
)

In [15]:
print("Telemetry shape:", telemetry.shape)
print("DAM shape:", dam.shape)

print("\nTelemetry:")
display(telemetry.head())

print("\nDAM:")
display(dam.head())

Telemetry shape: (1680, 5)
DAM shape: (336, 2)

Telemetry:


,timestamp,turbine_id,wind_speed,power_kw,availability
0,2026-03-01,T01,7.11,198.3,1.0
1,2026-03-01,T02,5.42,25.6,1.0
2,2026-03-01,T03,8.31,393.2,1.0
3,2026-03-01,T04,4.82,11.3,1.0
4,2026-03-01,T05,4.24,27.3,1.0



DAM:


,timestamp,dam_price_inr_per_kwh
0,2026-03-01 00:00:00,4.255
1,2026-03-01 01:00:00,4.330
2,2026-03-01 02:00:00,4.186
3,2026-03-01 03:00:00,4.032
4,2026-03-01 04:00:00,4.141


Basic Validation

In [16]:
print("Telemetry missing values:")
print(telemetry.isna().sum())

print("\nDAM missing values:")
print(dam.isna().sum())

Telemetry missing values:
timestamp       0
turbine_id      0
wind_speed      0
power_kw        0
availability    0
dtype: int64

DAM missing values:
timestamp                0
dam_price_inr_per_kwh    0
dtype: int64


In [17]:
print(
    "Duplicate telemetry rows:",
    telemetry.duplicated().sum()
)

print(
    "Duplicate DAM rows:",
    dam.duplicated().sum()
)

Duplicate telemetry rows: 0
Duplicate DAM rows: 0


In [18]:
print(
    "Telemetry time range:",
    telemetry["timestamp"].min(),
    "→",
    telemetry["timestamp"].max()
)

print(
    "DAM time range:",
    dam["timestamp"].min(),
    "→",
    dam["timestamp"].max()
)

Telemetry time range: 2026-03-01 00:00:00 → 2026-03-14 23:00:00
DAM time range: 2026-03-01 00:00:00 → 2026-03-14 23:00:00


In [19]:
import sqlite3
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Paths
# --------------------------------------------------

BASE_DIR = Path("..")

TURBINE_FILE = BASE_DIR / "data_pack" / "raw" / "turbine_telemetry_14day_sample.csv"
DAM_FILE = BASE_DIR / "data_pack" / "raw" / "dam_price_14day_sample.csv"

DB_FILE = BASE_DIR / "data_pack" / "greenko_part_a.db"


# --------------------------------------------------
# Load CSV files
# --------------------------------------------------

turbine_df = pd.read_csv(TURBINE_FILE)
dam_df = pd.read_csv(DAM_FILE)

print("Turbine data:")
display(turbine_df.head())

print("\nDAM data:")
display(dam_df.head())


# --------------------------------------------------
# Inspect columns
# --------------------------------------------------

print("Turbine columns:")
print(turbine_df.columns.tolist())

print("\nDAM columns:")
print(dam_df.columns.tolist())


# --------------------------------------------------
# Create SQLite database
# --------------------------------------------------

conn = sqlite3.connect(DB_FILE)


# --------------------------------------------------
# Write tables
# --------------------------------------------------

turbine_df.to_sql(
    "turbine_telemetry",
    conn,
    if_exists="replace",
    index=False
)

dam_df.to_sql(
    "dam_prices",
    conn,
    if_exists="replace",
    index=False
)


# --------------------------------------------------
# Verify tables
# --------------------------------------------------

print("\nTables created successfully.")

print(
    pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )
)


# --------------------------------------------------
# Test turbine table
# --------------------------------------------------

print("\nTurbine table:")
display(
    pd.read_sql(
        "SELECT * FROM turbine_telemetry LIMIT 5;",
        conn
    )
)


# --------------------------------------------------
# Test DAM table
# --------------------------------------------------

print("\nDAM table:")
display(
    pd.read_sql(
        "SELECT * FROM dam_prices LIMIT 5;",
        conn
    )
)


conn.close()

print(f"\nDatabase saved at: {DB_FILE}")

Turbine data:


,timestamp,turbine_id,wind_speed,power_kw,availability
0,2026-03-01 00:00:00,T01,7.11,198.3,1.0
1,2026-03-01 00:00:00,T02,5.42,25.6,1.0
2,2026-03-01 00:00:00,T03,8.31,393.2,1.0
3,2026-03-01 00:00:00,T04,4.82,11.3,1.0
4,2026-03-01 00:00:00,T05,4.24,27.3,1.0



DAM data:


,timestamp,dam_price_inr_per_kwh
0,2026-03-01 00:00:00,4.255
1,2026-03-01 01:00:00,4.330
2,2026-03-01 02:00:00,4.186
3,2026-03-01 03:00:00,4.032
4,2026-03-01 04:00:00,4.141


Turbine columns:
['timestamp', 'turbine_id', 'wind_speed', 'power_kw', 'availability']

DAM columns:
['timestamp', 'dam_price_inr_per_kwh']

Tables created successfully.
                name
0  turbine_telemetry
1         dam_prices

Turbine table:


,timestamp,turbine_id,wind_speed,power_kw,availability
0,2026-03-01 00:00:00,T01,7.11,198.3,1.0
1,2026-03-01 00:00:00,T02,5.42,25.6,1.0
2,2026-03-01 00:00:00,T03,8.31,393.2,1.0
3,2026-03-01 00:00:00,T04,4.82,11.3,1.0
4,2026-03-01 00:00:00,T05,4.24,27.3,1.0



DAM table:


,timestamp,dam_price_inr_per_kwh
0,2026-03-01 00:00:00,4.255
1,2026-03-01 01:00:00,4.330
2,2026-03-01 02:00:00,4.186
3,2026-03-01 03:00:00,4.032
4,2026-03-01 04:00:00,4.141



Database saved at: ..\data_pack\greenko_part_a.db
